# Assignment 1: DFS, BFS

## Breadth-First Search (BFS) and Depth-First Search (DFS)

**Name:** Farhan Asif  
**Roll No:** [24a12res919]  
**Course:** [CDA305- Artificial Intelligence Techniques Lab
]  
**Date:** 15 September 2026

## Objective

The objective of this assignment is to:<br>
`Compare Breadth First Search (BFS) and Depth First Search (DFS) with respect to the number
of steps required to reach the solution if they are reachable.`

The implementation will:

1. Represent the 3x3 grid puzzle as a state.
2. Generate valid neighboring states.
3. Check whether a puzzle is solvable.
4. Implement BFS.
5. Implement DFS.
6. Find the solution path from an initial state to the goal state.
7. Compare BFS and DFS based on solution depth, nodes explored, and execution time.

## Problem Description

The 3x3 grid puzzle consists of eight numbered tiles and one blank space
arranged on a 3 × 3 board.

The objective is to move the tiles until the following goal state
is reached:

1 2 3<br>
4 5 6<br>
7 8 B

where `B` represents the blank space.

A tile can be moved into the blank space only if the tile is directly
adjacent to it.

## SOLUTION

In [1]:
# The target state we have to reach if reachable
GOAL = (
    1,2,3,
    4,5,6,
    7,8,"B"
)

### Generating a random grid of 3x3 shape containing numbers from 1 to 8 and<br>a blank space `B`

In [2]:
from random import shuffle

def generate_random_state():
  "Return random 3x3 grid puzzle tuple."

  # Convert the GOAL into a list to shuffle.
  state_list = list(GOAL)

  # Rearrange state list
  shuffle(state_list)

  # Convert the shuffled list back into a tuple.
  random_state_tuple = tuple(state_list)

  # Return the randomly generated puzzle state.
  return random_state_tuple

### Counting inversions in a given state

Inversion means number of pairs where larger number appears before smaller number

In [3]:
def count_inversions(state):
  """Return number of inversions in a given state."""

  number_list =[]

  # Removing blank space 'B' from a given state and
  # appending all the elements to a list 'numbers_list'.
  for x in state:
    if x != "B":
      number_list.append(x)

  inversions = 0

  # Counting inversions in a list 'numbers_list'
  for i in range(len(number_list)):
    for j in range(i+1, len(number_list)):
      if number_list[i] > number_list[j]:
        # Increasing inversions by 1 by checking
        # if the current number is larger than its next number
        inversions += 1

  return inversions

 ### Checking if the given state is solvable
 - If the inversion is even, given state is solvable

In [4]:
def is_solvable(state):
  """Return True if 3x3 puzzle is solvable."""

  inversions = count_inversions(state)

  # If the inversions is even, given state is solvable otherwise unsolvable
  if inversions % 2 == 0:
    return True
  else:
    return False

### Checking which neighboring states can be reached by moving the blank space <br>in 3x3 grid puzzle for a given state:

- Possible neighboring states are returned by the function `get_neighbors`<br> in the following **order: [left, up, down, right]**

- `get_neighbors` is created in the **above order** to follow convention of exploring the left-most move first in both `BFS` and `DFS` :
  - In `BFS`: we use `get_neighbors(state)`
  - In `DFS`: we use `reversed(get_neighbors(state))`

In [5]:
def get_neighbors(state):
  """Return neighboring states in order: left, up, down, right."""

  # Find the position of the blank space.
  blank = state.index("B")

  # Get the row
  row = blank // 3

  # Get the column
  col = blank % 3

  # Store all possible neighboring states.
  neighbors = []

  # Move blank space left if it is not the left most column
  if col > 0:
    new_state = move_blank(state, blank, "left")
    neighbors.append(new_state)

  # Move blank space up if it is not the top most column
  if row > 0:
    new_state = move_blank(state, blank, "up")
    neighbors.append(new_state)

  # Move blank space down if it is not the down most column
  if row < 2:
    new_state = move_blank(state, blank, "down")
    neighbors.append(new_state)

  # Move blank space right if it is not the right most column
  if col < 2:
    new_state = move_blank(state, blank, "right")
    neighbors.append(new_state)

  return neighbors

### Moving the blank space
- We swap the blank space with the adjacent tile to create a new state.

In [6]:
def move_blank(state, blank, direction):
  "Return a new state after a move in the 3x3 puzzle."

  # Set the index of the tiles in rearranged state
  if direction == "up":
    move = -3
  elif direction == "down":
    move = 3
  elif direction == "right":
    move = 1
  elif direction == "left":
    move = -1

  # Convert the tuple to a list so that its elements can be modified.
  new_state = list(state)

  # Swapping the blank space with the adjacent tile
  new_state[blank], new_state[blank + move] = (
    new_state[blank + move],
    new_state[blank]
  )

  return tuple(new_state)

### Breadth-First Search (BFS)

- We are using `deque` class from the `collections` library.
- `deque` is *data structure* designed to efficiently perform actions(adding/removing) to both ends.

  - `frontier` is a `deque` object, and we are using it like a queue, which follows **FIFO (First In, First Out)**.

- `visited` is a **set**, we are using a set instead of a **list** because:

  - Searching for an element in a **set** has an average time complexity of **O(1)**, while searching for an element in a **list** has a time complexity of **O(n)**.

  - A **set** does not allow duplicates, so it prevents us from adding the same state reapeatedly.

- `parent` is a **dictionary** that stores the *predecessor (parent)state* of each *discovered state*. This allows us to reconstruct the *solution path* by backtracking from the `GOAL` to the initial state.

- `level` is a **dictionary** created to keep track of *number of moves* required to reach each state from the initial state.

- `state_explored` is created to keep track of how many different state explored by the algorithm.

#### BFS Process

- `frontier` initially contains the `initial_state`.

- The `while frontier:` loop continues as long as there are states waiting to be explored.

- `frontier.popleft()` removes the **oldest state**(left-most state in frontier) from the queue, following the **FIFO** principle.

- Each removed state is counted as an **explored state**.

- If the current state is the `GOAL`, the search stops.

- Otherwise, `get_neighbors(state)` generates all possible neighboring states that can be reached by moving the blank space.

- For each neighbor:
  - If the neighbor has already been discovered, it is skipped.
  - Otherwise, it is added to `visited`.
  - Its parent is stored in `parent`.
  - Its level is calculated as one more than the current state's level.
  - It is added to the end of `frontier`.

- `level[neighbor] = level[state] + 1` because the neighbor can be reached from the current state using exactly **one additional move**.

#### Checking for a Solution

- After the search ends, we check whether `GOAL` is present in `parent`.

- If `GOAL` is not in `parent`, it means the goal was never discovered, so no solution was found.

  - For a 3x3 puzzle, this can happen when the initial state is **unsolvable**.

- `moves = level[GOAL]` gives the number of moves required to reach the goal from the initial state.

#### Reconstructing the Solution Path

- The solution path is reconstructed by **backtracking from the `GOAL`** using the `parent` dictionary.

- Starting from `GOAL`, we repeatedly access: `parent[state]` until we reach `None`, which represents the initial state.

- The states are initially collected in reverse order:

  `GOAL → ... → initial_state`

- `path.reverse()` reverses the list to obtain the solution order:

  `initial_state → ... → GOAL`

- Finally, the function returns:
  - `path` → the sequence of states in the solution.
  - `moves` → the number of moves in the solution.
  - `state_explored` → the number of states explored by BFS.

In [7]:
from collections import deque

def bfs(initial_state):
  """Return path, moves using BFS."""

  frontier = deque([initial_state])  # A queue (FIFO)
  visited = {initial_state}          # A set (does not allow duplicates)
  parent = {initial_state: None}

  # Store the number of moves required to reach each state from the initial state.
  level = {initial_state: 0}

  # Count how many states are actually explored.
  state_explored = 0

  # Run the while loop until no states are waiting to be explored.
  while frontier:

    # Remove the oldest state from the queue (FIFO).
    state = frontier.popleft()

    # Count this state as explored.
    state_explored += 1

    # Stop searching when the goal is reached.
    if state == GOAL:
        break

    for neighbor in get_neighbors(state):

      # Only check states that have not been discovered before.
      if neighbor not in visited:

        # Mark the neighbor as discovered.
        visited.add(neighbor)

        # Store the state from which this neighbor was reached.
        parent[neighbor] = state

        # The neighbor is one move farther from the current state.
        level[neighbor] = level[state] + 1

        # Add the neighbor to the end of the queue.
        frontier.append(neighbor)

  # If the goal is not discovered, no solution exists.
  if GOAL not in parent:
    return None

  # Number of moves in the solution.
  moves = level[GOAL]

  # Reconstruct the path by backtracking from the goal to the initial state.
  path = []

  while state is not None:
    path.append(state)
    state = parent[state]

  # Reverse it to get solution Path.
  path.reverse()

  # Return the solution path, number of moves, number of states explored.
  return path, moves, state_explored

### Depth-First Search (DFS)

- `frontier` is a `list`, and we are making it behave like a stack, which follows **LIFO (Last In, First Out)**.

- `visited`, `parent`, `level`, `state_explored` are same as in **BFS**.

#### DFS Process

- `frontier` initially contains the `initial_state`.

- The `while frontier:` loop continues as long as there are states waiting to be explored.

- `frontier.pop()` removes the **newest state** from the stack, following the **LIFO** principle.

- Each removed state is counted as an **explored state**.

- If the current state is the `GOAL`, the search stops.

- Otherwise, `get_neighbors(state)` generates all possible neighboring states that can be reached by moving the blank space.

- For each neighbor:
  - If the neighbor has already been discovered, it is skipped.
  - Otherwise, it is added to `visited`.
  - Its parent is stored in `parent`.
  - Its level is calculated as one more than the current state's level.
  - - It is added to the **top of the stack** using `frontier.append(neighbor)`.

- `level[neighbor] = level[state] + 1` because the neighbor can be reached from the current state using exactly **one additional move**.

#### Maintaining Left-First Exploration

- `get_neighbors(state)` returns neighboring states in the following order: `[left, up, down, right]`

- Since DFS uses a stack, the **last state added is explored first**.

- Therefore, we use: `reversed(get_neighbors(state))`

- This causes the neighbors to be added to the stack in the order: `[right, down, up, left]`

- Because the stack follows **LIFO**, `left` is then removed and explored first.

#### Checking for a Solution

- After the search ends, we check whether `GOAL` is present in `parent`.

- If `GOAL` is not in `parent`, it means the goal was never discovered, so no solution was found.

  - For a 3x3 puzzle, this can happen when the initial state is **unsolvable**.

- `moves = level[GOAL]` gives the number of moves required to reach the goal from the initial state.

#### Reconstructing the Solution Path

- The solution path is reconstructed by **backtracking from the `GOAL`** using the `parent` dictionary.

- Starting from `GOAL`, we repeatedly access: `parent[state]` until we reach `None`, which represents the initial state.

- The states are initially collected in reverse order: `GOAL → ... → initial_state`

- `path.reverse()` reverses the list to obtain the solution order:

  `initial_state → ... → GOAL`

- Finally, the function returns:
  - `path` → the sequence of states in the solution.
  - `moves` → the number of moves in the solution.
  - `state_explored` → the number of states explored by DFS.

In [8]:
def dfs(initial_state):
  """Return path, moves using dfs."""

  # Using a list as a stack. A stack follows LIFO (Last In, First Out).
  frontier = [initial_state]

  visited = {initial_state}      # A set (does not allow duplicates)

  # Store the parent (predecessor) of each discovered state.
  parent = {initial_state: None}

  # Store the number of moves required to reach each state from the initial state.
  level = {initial_state: 0}

  # Count how many states are actually explored.
  state_explored = 0

  # Run the while loop until no states are waiting to be explored.
  while frontier:

    # Remove the newest state from the stack (LIFO).
    state = frontier.pop()

    # Count this state as explored.
    state_explored += 1

    # Stop searching when the goal is reached.
    if state == GOAL:
        break

    # Reverse the neighbor order so DFS explores the left-most move first.
    for neighbor in reversed(get_neighbors(state)):

      # Only check states that have not been discovered before.
      if neighbor not in visited:

        # Mark the neighbor as discovered.
        visited.add(neighbor)

        # Store the state from which this neighbor was reached.
        parent[neighbor] = state

        # The neighbor is one move farther from the current state.
        level[neighbor] = level[state] + 1

        # Push the neighbor to the top of the stack.
        frontier.append(neighbor)

  # If the goal is not discovered, no solution exists.
  if GOAL not in parent:
    return None

  # Number of moves in the solution.
  moves = level[GOAL]

  # Reconstruct the path by backtracking from the goal to the initial state.
  path = []

  while state is not None:
    path.append(state)
    state = parent[state]

  # Reverse it to get solution Path.
  path.reverse()

  # Return the solution path, number of moves, number of states explored.
  return path, moves, state_explored

### Display a  3x3 grid puzzle

In [9]:
def print_puzzle_state(state):
    """Display the state_tuple in 3x3 puzzle."""

    for i in range(0, 9, 3):

      # Print 3 elements at a time
        print(state[i:i + 3])

### Executing BFS and DFS

In [19]:
start = (
    1, 2, 3,
    4, 5, 6,
    "B", 7, 8
)

bfs_path, bfs_moves, bfs_explored = bfs(start)

print("Initial state:")
print_puzzle_state(start)

print("\nBFS solution path:")

for state in bfs_path:
    print_puzzle_state(state)
    print()

Initial state:
(1, 2, 3)
(4, 5, 6)
('B', 7, 8)

BFS solution path:
(1, 2, 3)
(4, 5, 6)
('B', 7, 8)

(1, 2, 3)
(4, 5, 6)
(7, 'B', 8)

(1, 2, 3)
(4, 5, 6)
(7, 8, 'B')



In [10]:
from time import perf_counter

# Generate a random puzzle state(does not guarantee a solvable state).
start = generate_random_state()

# Track whether the generated state is solvable.
solvable = False

# Check whether the generated puzzle state is solvable.
if is_solvable(start):

  # Print the initial puzzle state.
  print(f"Initial state:")
  print_puzzle_state(start)

  solvable = True

else:

  # Print the initial puzzle state.
  print_puzzle_state(start)
  print("Initial state is unsolvable.")

  solvable = False

# Run BFS and DFS only if the puzzle is solvable.
if solvable:

  # Start the timer before running BFS.
  start_time = perf_counter()

  # Run BFS and store the solution path, number of moves,
  # and number of states explored.
  bfs_path, bfs_moves, no_of_state_explored_using_bfs = bfs(start)

  # Calculate the total time taken by BFS.
  bfs_time = perf_counter() - start_time


  # Start a new timer before running DFS.
  start_time = perf_counter()

  # Run DFS and store the solution path, number of moves,
  # and number of states explored.
  dfs_path, dfs_moves, no_of_state_explored_using_dfs = dfs(start)

  # Calculate the total time taken by DFS.
  dfs_time = perf_counter() - start_time

  # Print the BFS results.
  print((f"\nBFS\n"
  f"Moves: {bfs_moves}\n"
  f"state_explored: {no_of_state_explored_using_bfs}\n"
  f"Time: {bfs_time}"))

  # Print the DFS results.
  print((f"\nDFS\n"
  f"Moves: {dfs_moves}\n"
  f"state_explored: {no_of_state_explored_using_dfs}\n"
  f"Time: {dfs_time}"))

Initial state:
(5, 1, 3)
(2, 4, 'B')
(8, 7, 6)

BFS
Moves: 17
state_explored: 12857
Time: 0.1060126369993668

DFS
Moves: 62197
state_explored: 113561
Time: 1.190447109998786


#### Examples of Unsolvable states

unsolvable_states = [<br>
    (8, 7, 3,<br>
    4, 'B', 1,<br>
    6, 2, 5)<br>
,<br>
    (2, 8, 7,<br>
    3, 6, 4,<br>
    'B', 1, 5)<br>
,<br>
    (1, 6, 8,<br>
    'B', 5, 3,<br>
    7, 2, 4)<br>
,<br>...
      ]

#### Examples of Solvable states

solvable_states = [<br>
    (6, 7, 2,<br>
    'B', 4, 3,<br>
    8, 1, 5)<br>
,<br>
    (1, 5, 'B',<br>
    4, 2, 3,<br>
    8, 7, 6)<br>
,<br>
    ('B', 2, 6,<br>
    5, 8, 7,<br>
    3, 1, 4)<br>
,<br>
  ...    ]

#### Always_Solvable_random_state_generator

In [11]:
def generate_random_solvable_state():
  "Return random 3x3 grid puzzle tuple."

  # Run the while loop until there is a solvable state
  while True:
    # Convert the GOAL into a list to shuffle.
    state_list = list(GOAL)

    # Rearrange state list
    shuffle(state_list)

    # Convert the shuffled list back into a tuple.
    random_state_tuple = tuple(state_list)

    if is_solvable(random_state_tuple):
      # Return the randomly generated solvable puzzle state.
      return random_state_tuple


In [15]:
from time import perf_counter

# Generate a random puzzle solvable state(guarantee a solvable state).
start = generate_random_solvable_state()

# Start the timer before running BFS.
start_time = perf_counter()

# Run BFS and store the solution path, number of moves,
# and number of states explored.
bfs_path, bfs_moves, no_of_state_explored_using_bfs = bfs(start)

# Calculate the total time taken by BFS.
bfs_time = perf_counter() - start_time


# Start a new timer before running DFS.
start_time = perf_counter()

# Run DFS and store the solution path, number of moves,
# and number of states explored.
dfs_path, dfs_moves, no_of_state_explored_using_dfs = dfs(start)

# Calculate the total time taken by DFS.
dfs_time = perf_counter() - start_time

print(f"Initial state:")
print_puzzle_state(start)

# Print the BFS results.
print((f"\n-------BFS-------\n"
f"Moves: {bfs_moves}\n"
f"state_explored: {no_of_state_explored_using_bfs}\n"
f"Time: {bfs_time}"))

# Print the DFS results.
print((f"\n-------DFS-------\n"
f"Moves: {dfs_moves}\n"
f"state_explored: {no_of_state_explored_using_dfs}\n"
f"Time: {dfs_time}"))

Initial state:
(2, 5, 6)
(3, 'B', 1)
(4, 7, 8)

-------BFS-------
Moves: 16
state_explored: 9457
Time: 0.03932368000096176

-------DFS-------
Moves: 35232
state_explored: 146331
Time: 0.8538382090009691


#### 10-TRIAL EXPERIMENT

In [17]:
# Run the experiment 10 times
for i in range(10):

  # Generate a new random solvable puzzle for each trial
  start = generate_random_solvable_state()

  # Run BFS and measure its execution time
  start_time = perf_counter()
  bfs_path, bfs_moves, bfs_explored = bfs(start)
  bfs_time = perf_counter() - start_time

  # Run DFS and measure its execution time
  start_time = perf_counter()
  dfs_path, dfs_moves, dfs_explored = dfs(start)
  dfs_time = perf_counter() - start_time

  # Display the trial number
  print(f"\n========== Trial {i + 1} ==========")

  # Display the initial puzzle state
  print("Initial state:")
  print_puzzle_state(start)

  # Display BFS results
  print(f"\nBFS")
  print(f"Moves: {bfs_moves}")
  print(f"States explored: {bfs_explored}")
  print(f"Time: {bfs_time:.6f} seconds")

  # Display DFS results
  print(f"\nDFS")
  print(f"Moves: {dfs_moves}")
  print(f"States explored: {dfs_explored}")
  print(f"Time: {dfs_time:.6f} seconds")


========== Trial 1 ==========
Initial state:
(3, 8, 2)
('B', 6, 5)
(7, 1, 4)

BFS
Moves: 25
States explored: 154798
Time: 0.629012 seconds

DFS
Moves: 69743
States explored: 95704
Time: 0.471661 seconds

========== Trial 2 ==========
Initial state:
(3, 7, 4)
(1, 6, 2)
('B', 8, 5)

BFS
Moves: 18
States explored: 23924
Time: 0.095829 seconds

DFS
Moves: 47886
States explored: 54730
Time: 0.283601 seconds

========== Trial 3 ==========
Initial state:
(6, 3, 4)
(2, 7, 5)
(8, 1, 'B')

BFS
Moves: 24
States explored: 138288
Time: 0.566770 seconds

DFS
Moves: 68636
States explored: 100237
Time: 0.446921 seconds

========== Trial 4 ==========
Initial state:
(5, 7, 'B')
(3, 1, 4)
(6, 8, 2)

BFS
Moves: 26
States explored: 167013
Time: 1.007944 seconds

DFS
Moves: 11914
States explored: 170248
Time: 1.114438 seconds

========== Trial 5 ==========
Initial state:
(7, 'B', 8)
(5, 6, 4)
(2, 1, 3)

BFS
Moves: 25
States explored: 156539
Time: 1.032078 seconds

DFS
Moves: 59377
States explored: 69791
Ti

##### **10-TRIAL Results**
========== Trial 1 ==========<br>
Initial state:<br>
(3, 8, 2)<br>
('B', 6, 5)<br>
(7, 1, 4)<br>

------BFS------<br>
Moves: 25<br>
States explored: 154798<br>
Time: 0.629012 seconds<br>
<br>
------DFS------<br>
Moves: 69743<br>
States explored: 95704<br>
Time: 0.471661 seconds<br>
<br><br>
========== Trial 2 ==========<br>
Initial state:<br>
(3, 7, 4)<br>
(1, 6, 2)<br>
('B', 8, 5)<br>
<br>
------BFS------<br>
Moves: 18<br>
States explored: 23924<br>
Time: 0.095829 seconds<br>
<br>
------DFS------<br>
Moves: 47886<br>
States explored: 54730<br>
Time: 0.283601 seconds<br>
<br><br>
========== Trial 3 ==========<br>
Initial state:<br>
(6, 3, 4)<br>
(2, 7, 5)<br>
(8, 1, 'B')<br>
<br>
------BFS------<br>
Moves: 24<br>
States explored: 138288<br>
Time: 0.566770 seconds<br>
<br>
------DFS------<br>
Moves: 68636<br>
States explored: 100237<br>
Time: 0.446921 seconds<br>
<br><br>
========== Trial 4 ==========
Initial state:<br>
(5, 7, 'B')<br>
(3, 1, 4)<br>
(6, 8, 2)<br>
<br>
------BFS------<br>
Moves: 26<br>
States explored: 167013<br>
Time: 1.007944 seconds<br>
<br>
------DFS------<br>
Moves: 11914<br>
States explored: 170248<br>
Time: 1.114438 seconds<br>
<br><br>
========== Trial 5 ==========<br>
Initial state:<br>
(7, 'B', 8)<br>
(5, 6, 4)<br>
(2, 1, 3)<br>
<br>
------BFS------<br>
Moves: 25<br>
States explored: 156539<br>
Time: 1.032078 seconds<br>
<br>
------DFS------<br>
Moves: 59377<br>
States explored: 69791<br>
Time: 0.408537 seconds<br>
<br><br>
========== Trial 6 ==========<br>
Initial state:<br>
(7, 6, 3)<br>
(2, 'B', 8)<br>
(1, 5, 4)<br>
<br>
------BFS------<br>
Moves: 18<br>
States explored: 25248<br>
Time: 0.100441 seconds<br>
<br>
------DFS------<br>
Moves: 69020<br>
States explored: 98680<br>
Time: 0.454773 seconds<br>
<br><br>
========== Trial 7 ==========<br><br>
Initial state:<br>
(5, 8, 4)<br>
(3, 'B', 6)<br>
(2, 1, 7)<br>
<br>
------BFS------<br>
Moves: 22<br>
States explored: 90053<br>
Time: 0.365799 seconds<br>
<br>
------DFS------<br>
Moves: 68914<br>
States explored: 87001<br>
Time: 0.423405 seconds<br>
<br><br>
========== Trial 8 ==========<br>
Initial state:<br>
(7, 3, 'B')<br>
(8, 6, 5)<br>
(4, 1, 2)<br>
<br>
------BFS------<br>
Moves: 20<br>
States explored: 52349<br>
Time: 0.201017 seconds<br>
<br>
------DFS------<br>
Moves: 12246<br>
States explored: 169921<br>
Time: 0.692079 seconds<br>
<br>
========== Trial 9 ==========<br>
Initial state:<br>
('B', 3, 1)<br>
(2, 4, 8)<br>
(7, 6, 5)<br>
<br>
------BFS------<br>
Moves: 20<br>
States explored: 46403<br>
Time: 0.173116 seconds<br>
<br>
------DFS------<br>
Moves: 350<br>
States explored: 181126<br>
Time: 0.673250 seconds<br>
<br><br>
========== Trial 10 ==========<br>
Initial state:<br>
(3, 'B', 8)<br>
(2, 4, 1)<br>
(7, 5, 6)<br>
<br>
------BFS------<br>
Moves: 21<br>
States explored: 64075<br>
Time: 0.254304 seconds<br>
<br>
------DFS------<br>
Moves: 38173<br>
States explored: 143001<br>
Time: 0.566897 seconds<br>
<br>

##### ------**AVERAGE**------<br>
------BFS------<br>
Moves: 21.9<br>
States explored: 91869<br>
Time: 0.442631 seconds<br>


------DFS------<br>
Moves: 44625.9<br>
States explored: 117043.9<br>
Time: 0.553556 seconds<br>


#### Q2. ***Comment on which algorithm will be faster and when, by mentioning proper intuition and examples.***

BFS will generally be faster when the goal is reached after exploring relatively few levels, because it explores states level by level. For example, if the goal is only 2 moves away, BFS reaches it after checking the states at depths 0, 1, and 2.

DFS can be faster when the goal lies along the first branch it explores. For example, if DFS's first explored branch reaches the goal quickly, it may find the solution without exploring many other branches.

Therefore, the faster algorithm depends on the starting state and search order. In our 10 trials, BFS was faster in 7 trials and DFS was faster in 3 trials. BFS also had the lower average execution time (0.443 s vs 0.554 s).

## Experimental Results and Analysis

BFS and DFS were tested on 10 randomly generated solvable 3x3 grid puzzle states.

The results show that **BFS consistently produced much shorter solutions than DFS**. BFS required between 18 and 26 moves, with an average of 21.9 moves. DFS required between 350 and 69,743 moves, with an average of 44,625.9 moves.

BFS also explored fewer states on average. BFS explored an average of 91,869 states, while DFS explored an average of 117,043.9 states.

The average execution time was 0.443 seconds for BFS and 0.554 seconds for DFS. BFS was faster in most of the trials, but DFS was faster in some trials. Therefore, ***the experimental results do not support the claim that BFS is always faster than DFS***.

**The main difference is that BFS explores states level by level. Since every move in the 8-puzzle has the same cost, BFS finds the solution requiring the minimum number of moves. DFS explores deeply according to its search order and therefore can find a solution that is much longer than the optimal solution.**

## BFS vs DFS Comparison

### Breadth-First Search (BFS)

BFS uses a ***FIFO*** *queue* and explores the puzzle states level by level.<br>
For the 3x3 grid puzzle, each move has equal cost,
<br>**so the first solution found by BFS has the minimum number of moves**.
<br><br>

In all 10 experiments, BFS found solutions between 18 and 26 moves. <br>
Therefore, **BFS was optimal for these experiments**.
<br><br>

**The disadvantage of BFS is memory usage**. <br>
It can store a large number of states in the frontier while exploring the puzzle level by level.
<br><br><br>

### Depth-First Search (DFS)

DFS uses a ***LIFO*** *stack* and explores one branch deeply before exploring other branches. <br>
**It does not guarantee the shortest solution**.
<br><br>

In the experiments, DFS found solutions that were much longer than the BFS solutions. <br>
This demonstrates that DFS can spend a large amount of search effort following a deep path before reaching the goal.
<br><br>

**DFS generally requires less frontier memory than BFS**,<br>
although its actual execution time can vary depending on the starting state and search order.
<br><br><br><br>

### Overall Comparison

**BFS is preferable when finding the shortest solution is important.**<br>
**DFS can be preferable when memory usage is more important and an optimal solution is not required.**
<br><br>

The experiments also show that execution time is dependent on the particular starting state. **BFS was not faster in every individual trial**.

## Conclusion

The 3x3 grid puzzle was successfully solved using both Breadth-First Search (BFS) and Depth-First Search (DFS).

The puzzle state was represented as a *tuple* containing the numbers 1-8 and the blank space `B`. <br>
Valid successor states were generated by moving the blank space up, down, left, or right. <br>
An *inversion-counting method* was used to ensure that the randomly generated initial state was solvable.

BFS used a queue and explored states level by level. <br>
Because every puzzle move has equal cost, BFS found the shortest solution in every experiment.

DFS used a stack and explored states depth-first. <br>
It successfully found solutions, but it did not guarantee the shortest solution and produced significantly longer paths in the experiments.

**Across 10 trials, BFS required an average of 21.9 moves, while DFS required an average of 44,625.9 moves.** <br>
BFS also explored fewer states on average and had a lower average execution time.

Therefore, **BFS is more suitable for the 3x3 grid puzzle when the objective is to find the shortest solution.**<br>
**DFS can use less frontier memory**, but its solution quality and execution time depend strongly on the search order and starting state.